# Aim for max movement of < 1.27 mm (row pitch) ?
 - Cos max pressure is in correct location so don't want to go to far
 - Also it would be visibly obvious if they were much futher off?

 ---
 - registration errors are around 0.5 mm and 0.5 degrees

In [65]:
import numpy as np
import pyvista as pv
from pathlib import Path

from phd_helpers.paths import transform_mesh, get_bone_inertia, get_subject_stl_path, transform_points

In [ ]:
sub = '14548R'
path = Path('../../../../../Computational/MeshPipeline/outputs/initialFEAstuff/35T/35Tbest')
tpm_path = list(path.glob(f'**/{sub}/tpm-mc1/**/*.vtu'))[0]
mc1_path = list(path.glob(f'**/{sub}/mc1-tpm/**/*.vtu'))[0]

tpm = pv.read(tpm_path)
mc1 = pv.read(mc1_path)

# transform to mc1 coordinate system 
# - so that movements are relative to mc1 inertial axes - for consistency
stl_path = get_subject_stl_path(sub[:-1], sub[-1])
mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
tpm_centroid, _, _ = get_bone_inertia(stl_path, 'tpm')

tpm_mc1 = transform_mesh(tpm, mc1_axes, mc1_centroid, inverse=True)
mc1_mc1 = transform_mesh(mc1, mc1_axes, mc1_centroid, inverse=True)
tpm_centroid = transform_points(tpm_centroid, mc1_axes, mc1_centroid, inverse=True)[0]

In [93]:
R = 5 # degrees
t = 1.0 # mm
tpm_Rx = tpm_mc1.rotate_x(R, point=tpm_centroid, transform_all_input_vectors=True)
tpm_Ry = tpm_mc1.rotate_y(R, point=tpm_centroid, transform_all_input_vectors=True)
tpm_Rz = tpm_mc1.rotate_z(R, point=tpm_centroid, transform_all_input_vectors=True)

tpm_ty = tpm_mc1.translate([0., t, 0.], transform_all_input_vectors=True)
tpm_tz = tpm_mc1.translate([0., 0., t], transform_all_input_vectors=True)

# max point movement
np.linalg.norm(tpm_mc1.points - tpm_Rx.points, axis=1).max()

np.float64(0.9869448595145695)

In [96]:
mesh = tpm_Rx

pl = pv.Plotter()
pl.add_mesh(mc1_mc1, color='white')
pl.add_mesh(tpm_mc1, color='black', style='wireframe')
pl.add_mesh(mesh, color='white')
pl.show_axes()
pl.show()

Widget(value='<iframe src="http://localhost:54329/index.html?ui=P_0x373377710_30&reconnect=auto" class="pyvist…

In [95]:
# transform back to global

mesh_glob = transform_mesh(mesh, mc1_axes, mc1_centroid)

pl = pv.Plotter()
pl.add_mesh(mc1, color='white')
pl.add_mesh(tpm, color='black', style='wireframe')
pl.add_mesh(mesh_glob, color='white')
pl.show_axes()
pl.show()

Widget(value='<iframe src="http://localhost:54329/index.html?ui=P_0x373389e20_29&reconnect=auto" class="pyvist…

In [97]:
# save meshes in global 

tpm_savedir = path.parent / f'TwistTranslate1/meshes/{sub}/tpm-mc1/3Dmesh'
tpm_savedir.mkdir(parents=True, exist_ok=True)

mc1_savedir = path.parent / f'TwistTranslate1/meshes/{sub}/mc1-tpm/3Dmesh'
mc1_savedir.mkdir(parents=True, exist_ok=True)

names = ['Rx', 'Ry', 'Rz', 'ty', 'tz']
meshes = [tpm_Rx, tpm_Ry, tpm_Rz, tpm_ty, tpm_tz]

for mesh, name in zip(meshes, names):

    mesh_glob = transform_mesh(mesh, mc1_axes, mc1_centroid)
    mesh_glob.save(tpm_savedir / f'mesh-{name}.vtu')
    mc1.save(mc1_savedir / f'mesh-{name}.vtu')

# Verify .inp meshes are correct
 - They get translated along x slightly for inp file so just check other alignements

In [98]:
from phd_helpers.AbaqusPostprocessing import inp2pv

In [119]:
inp_path = Path('../../../../../Computational/InpPipeline/outputs/initialFEAstuff/accuracy/TwistTranslate/study1')
name = 'ty'
mesh = tpm_ty

inp_mesh = inp2pv(list(inp_path.glob(f'**/{name}*.inp'))[0])
inp_tpm = inp_mesh['tpm']
inp_mc1 = inp_mesh['mc1']

In [120]:
pl = pv.Plotter()
pl.add_mesh(inp_mc1, color='white')
pl.add_mesh(inp_tpm, color='white')
pl.add_mesh(mesh, color='black', style='wireframe')
pl.show_axes()
pl.show()

Widget(value='<iframe src="http://localhost:54329/index.html?ui=P_0x3732cd520_36&reconnect=auto" class="pyvist…

# Study 2

In [121]:
import numpy as np
import pyvista as pv
from pathlib import Path

from phd_helpers.paths import transform_mesh, get_bone_inertia, get_subject_stl_path, transform_points

In [123]:
R = -5 # degrees
t = -1.0 # mm

sub = '14548R'
mesh_out_root_name = 'TwistTranslate2'

In [124]:
# load meshes
path = Path('../../../../../Computational/MeshPipeline/outputs/initialFEAstuff/35T/35Tbest')
tpm_path = list(path.glob(f'**/{sub}/tpm-mc1/**/*.vtu'))[0]
mc1_path = list(path.glob(f'**/{sub}/mc1-tpm/**/*.vtu'))[0]

tpm = pv.read(tpm_path)
mc1 = pv.read(mc1_path)

# transform to mc1 coordinate system 
# - so that movements are relative to mc1 inertial axes - for consistency
stl_path = get_subject_stl_path(sub[:-1], sub[-1])
mc1_centroid, _, mc1_axes = get_bone_inertia(stl_path, 'mc1')
tpm_centroid, _, _ = get_bone_inertia(stl_path, 'tpm')

tpm_mc1 = transform_mesh(tpm, mc1_axes, mc1_centroid, inverse=True)
mc1_mc1 = transform_mesh(mc1, mc1_axes, mc1_centroid, inverse=True)
tpm_centroid = transform_points(tpm_centroid, mc1_axes, mc1_centroid, inverse=True)[0]

tpm_Rx = tpm_mc1.rotate_x(R, point=tpm_centroid, transform_all_input_vectors=True)
tpm_Ry = tpm_mc1.rotate_y(R, point=tpm_centroid, transform_all_input_vectors=True)
tpm_Rz = tpm_mc1.rotate_z(R, point=tpm_centroid, transform_all_input_vectors=True)

tpm_ty = tpm_mc1.translate([0., t, 0.], transform_all_input_vectors=True)
tpm_tz = tpm_mc1.translate([0., 0., t], transform_all_input_vectors=True)

# save meshes in global 
tpm_savedir = path.parent / f'{mesh_out_root_name}/meshes/{sub}/tpm-mc1/3Dmesh'
tpm_savedir.mkdir(parents=True, exist_ok=True)

mc1_savedir = path.parent / f'{mesh_out_root_name}/meshes/{sub}/mc1-tpm/3Dmesh'
mc1_savedir.mkdir(parents=True, exist_ok=True)

names = ['Rx', 'Ry', 'Rz', 'ty', 'tz']
meshes = [tpm_Rx, tpm_Ry, tpm_Rz, tpm_ty, tpm_tz]

for mesh, name in zip(meshes, names):

    mesh_glob = transform_mesh(mesh, mc1_axes, mc1_centroid)
    mesh_glob.save(tpm_savedir / f'mesh-{name}.vtu')
    mc1.save(mc1_savedir / f'mesh-{name}.vtu')

